# 호텔검색 보고서 수치 검증 노트북

2026-09-04 분석 보고서의 값을 SQLite에서 직접 재계산합니다. 코드 셀 아래에 결과가 저장되어 있어 실행 전에도 읽을 수 있습니다.

**사용법:** VS Code에서 오른쪽 위 커널을 `Python 3.14 (team_i)` 또는 라이브러리가 설치된 Python 3.14로 선택 → **모두 실행(Run All)**. 셀 하나씩은 `Shift+Enter`입니다. 중간 셀부터 실행하면 앞에서 정의한 변수가 없어 오류가 날 수 있습니다.

순서: 환경 → 데이터/결합 → A → B → C → D·F → G → H → A-2 통제모형 → 보고서 대조.

비율은 모두 분자/분모×100입니다. 기술통계는 6,900검색·1,000세션, 통제모형은 기존 보고서와 동일한 43개 대표 경로·296검색을 사용합니다. 예약·이탈·개입 효과는 필요한 데이터가 없으므로 이번 수치 검증 범위에 포함되지 않습니다.


## 1. 환경 확인

라이브러리가 없을 때만 다음 셀의 주석을 풀어 실행하고 커널을 재시작하세요. `%pip`는 현재 노트북 커널에 설치합니다.

In [1]:
# %pip install pandas numpy scipy statsmodels==0.15.0 ipykernel

In [2]:
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sqlite3
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf


TABLES = ["user", "hotel", "room", "search", "search_filter", "search_result", "event", "booking"]
TRANSITION_FIELDS = [
    "query_text", "destination", "property_type", "property_grade",
    "user_rating_min", "price", "amenity_count", "region",
]
SIGNATURE_FIELDS = [
    "query_text", "total_result_count", "sort_option", "guest_count",
    "destination", "property_type", "property_grade", "user_rating_min",
    "price", "amenity_count", "region",
]
from IPython.display import display
import sys
pd.set_option("display.max_columns", 20)
print("Python:", sys.executable)
print("pandas:", pd.__version__, "statsmodels:", statsmodels.__version__)

Python: c:\Users\ENCO\AppData\Local\Python\pythoncore-3.14-64\python.exe
pandas: 3.0.5 statsmodels: 0.15.0


## 2. 공통 함수

기존 분석 코드의 읽기 전용 연결, 비율 계산, 전이 분류 함수를 그대로 사용합니다. 전이는 동일조건 → 지역 변경 → 검색어 변경 → 완화/강화 순서로 판정합니다. 가격·평점 결측은 미설정이며 문자열 비교는 NFKC 정규화·공백 제거·소문자화합니다. 날짜·정렬·인원 변경은 기존 전이 분류 대상에서 제외되어 있습니다.

In [3]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def open_read_only(path: Path) -> sqlite3.Connection:
    connection = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
    connection.execute("PRAGMA query_only=ON")
    assert connection.execute("PRAGMA query_only").fetchone()[0] == 1
    return connection


def rate(numerator: int, denominator: int) -> float | None:
    return numerator / denominator if denominator else None


def metric(numerator: int, denominator: int, unit: str) -> dict:
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": rate(int(numerator), int(denominator)),
        "analysis_unit": unit,
    }


def normalize(value) -> str | None:
    if pd.isna(value):
        return None
    text = unicodedata.normalize("NFKC", str(value)).strip().casefold()
    return text or None


def direction(old, new, lower_is_relaxation: bool) -> str | None:
    old_missing, new_missing = pd.isna(old), pd.isna(new)
    if old_missing and new_missing:
        return None
    if not old_missing and new_missing:
        return "relax"
    if old_missing and not new_missing:
        return "strengthen"
    if float(old) == float(new):
        return None
    decreased = float(new) < float(old)
    relaxed = decreased if lower_is_relaxation else not decreased
    return "relax" if relaxed else "strengthen"


def classify_transition(row: pd.Series) -> str:
    def equal(field: str) -> bool:
        old, new = row[field], row[f"next_{field}"]
        if field in {"user_rating_min", "price", "amenity_count"}:
            if pd.isna(old) and pd.isna(new):
                return True
            if pd.isna(old) or pd.isna(new):
                return False
            return float(old) == float(new)
        return normalize(old) == normalize(new)

    changed = [field for field in TRANSITION_FIELDS if not equal(field)]
    if not changed:
        return "동일조건 반복"
    if "destination" in changed or "region" in changed:
        return "지역 변경"
    if "query_text" in changed:
        return "검색어 변경"
    directions = {
        item for item in [
            direction(row["price"], row["next_price"], False),
            direction(row["user_rating_min"], row["next_user_rating_min"], True),
            direction(row["amenity_count"], row["next_amenity_count"], True),
        ] if item
    }
    if directions == {"relax", "strengthen"}:
        return "완화·강화 혼합"
    if directions == {"relax"}:
        return "조건 완화"
    if directions == {"strengthen"}:
        return "조건 강화"
    raise ValueError(f"승인되지 않은 전이 유형: {changed}")


def session_signature(group: pd.DataFrame) -> str:
    # Stay dates are omitted because invalid source dates were repaired with sampled
    # valid durations; all other search/filter states recover the 43 source paths.
    payload = group[SIGNATURE_FIELDS].fillna("<NA>").astype(str).values.tolist()
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False).encode("utf-8")).hexdigest()

In [4]:
def show_metrics(items):
    rows = []
    for label, item in items.items():
        rows.append({"지표": label, "분자": item["numerator"], "분모": item["denominator"],
                     "비율(%)": 100 * item["rate"] if item["rate"] is not None else None,
                     "단위": item["analysis_unit"]})
    display(pd.DataFrame(rows).style.format({"비율(%)": "{:.2f}"}))

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / '00_프로젝트관리').is_dir() and (p / '09_단계별 분석').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('team_i 폴더 또는 그 하위 폴더에서 실행하세요.')
analysis_folder = ROOT / '04_분석설계/팀프로젝트/2026/09'
db = ROOT / '09_단계별 분석/2단계_1000건_증감_분석/02_관측형합성1000명_실행묶음_260904_1259_01/호텔검색_관측형합성1000명_데이터_260904_1259_01.sqlite'
reference_path = analysis_folder / '호텔검색_증강데이터인사이트분석결과_20260904_v01_현행본.json'
reference = json.loads(reference_path.read_text(encoding='utf-8'))
assert db.is_file(), db
print('입력 SQLite:', db)
print('대조 JSON:', reference_path)

입력 SQLite: c:\Users\ENCO\Documents\mission1\team_i\09_단계별 분석\2단계_1000건_증감_분석\02_관측형합성1000명_실행묶음_260904_1259_01\호텔검색_관측형합성1000명_데이터_260904_1259_01.sqlite
대조 JSON: c:\Users\ENCO\Documents\mission1\team_i\04_분석설계\팀프로젝트\2026\09\호텔검색_증강데이터인사이트분석결과_20260904_v01_현행본.json


## 3. SQLite 로딩과 전처리

SEARCH–SEARCH_FILTER를 `search_id`로 1:1 결합하고, 세션·검색 시각·검색 ID로 정렬합니다. 순위 클릭은 고유 `(search_id, hotel_id)`로 집계합니다. DB 내용은 변경하지 않습니다.

In [5]:
before_hash = sha256(db)
connection = open_read_only(db)
counts = {
    table: connection.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
    for table in TABLES
}
integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
search = pd.read_sql_query("SELECT * FROM search", connection)
search_filter = pd.read_sql_query("SELECT * FROM search_filter", connection)
search_result = pd.read_sql_query("SELECT search_id,hotel_id,result_rank FROM search_result", connection)
event = pd.read_sql_query(
    "SELECT event_id,session_id,event_type,event_at,hotel_id,search_id FROM event",
    connection,
)

assert search.search_id.is_unique
assert search_filter.search_id.is_unique
assert set(search.search_id) == set(search_filter.search_id)
base = search.merge(search_filter, on="search_id", how="inner", validate="one_to_one", suffixes=("", "_filter"))
base["search_at"] = pd.to_datetime(base.search_time.str.replace(" KST", "", regex=False), errors="raise")
base = base.sort_values(["session_id", "search_at", "search_id"], kind="stable").reset_index(drop=True)
base["search_order"] = base.groupby("session_id").cumcount() + 1
base["zero"] = base.total_result_count.eq(0)
lead_fields = ["search_id", "total_result_count", *TRANSITION_FIELDS]
for field in lead_fields:
    base[f"next_{field}"] = base.groupby("session_id", sort=False)[field].shift(-1)
base["has_next"] = base.next_search_id.notna()

click_events = event[event.event_type.eq("hotel_click") & event.search_id.notna() & event.hotel_id.notna()]
click_search_ids = set(click_events.search_id)
click_pairs = set(click_events[["search_id", "hotel_id"]].itertuples(index=False, name=None))
display(pd.DataFrame(list(counts.items()), columns=['테이블', '행 수']))
print('무결성:', integrity)
print('검색 기간(KST):', base.search_at.min(), '~', base.search_at.max())
print('분석 전 SHA-256:', before_hash)
assert before_hash == reference['analysis']['database_sha256_before'], '보고서와 다른 DB입니다.'
display(base[['search_id','session_id','search_order','total_result_count','price','user_rating_min','amenity_count']].head(10))

,테이블,행 수
0,user,1000
1,hotel,1000
2,room,3000
3,search,6900
4,search_filter,6900
5,search_result,198128
6,event,238851
7,booking,0


무결성: ok
검색 기간(KST): 2027-01-01 00:01:00 ~ 2027-09-08 00:02:30
분석 전 SHA-256: d30f40b9d0c85a8f8469f48b37daae5f6dc3e4a77eea81671310dda7cbd8e79d


,search_id,session_id,search_order,total_result_count,price,user_rating_min,amenity_count
0,SYN_Q0001_001,SYN_S0001,1,0,NaN,NaN,0
1,SYN_Q0001_002,SYN_S0001,2,0,200000.0,8.0,7
2,SYN_Q0001_003,SYN_S0001,3,0,200000.0,8.0,7
3,SYN_Q0001_004,SYN_S0001,4,0,200000.0,8.0,7
4,SYN_Q0001_005,SYN_S0001,5,116,200000.0,NaN,0
5,SYN_Q0002_001,SYN_S0002,1,10,NaN,NaN,0
6,SYN_Q0002_002,SYN_S0002,2,2,NaN,NaN,0
7,SYN_Q0002_003,SYN_S0002,3,0,NaN,9.0,0
8,SYN_Q0002_004,SYN_S0002,4,0,NaN,9.0,2
9,SYN_Q0003_001,SYN_S0003,1,10,NaN,NaN,0


## 4. 카드 A — 제한조건별 0건률

분자: 해당 조건이 설정된 검색 중 결과 0건. 분모: 해당 조건이 설정된 전체 검색. 조건들은 중첩될 수 있습니다.

In [6]:
card_a = {}
conditions = {
    "amenity_count_ge_3": base.amenity_count.ge(3),
    "minimum_rating_set": base.user_rating_min.notna(),
    "price_filter_set": base.price.notna(),
}
for name, mask in conditions.items():
    card_a[name] = metric(int((mask & base.zero).sum()), int(mask.sum()), "search")
show_metrics(card_a)

,지표,분자,분모,비율(%),단위
0,amenity_count_ge_3,2808,3181,88.27,search
1,minimum_rating_set,2695,3558,75.74,search
2,price_filter_set,2473,3408,72.56,search


수치 해석: 보고서의 편의시설 2,808/3,181, 평점 2,695/3,558, 가격 2,473/3,408을 대조합니다. 조건별 집단이 겹치므로 이 차이만으로 조건 해제의 효과를 확정할 수 없습니다.

## 5. 카드 B — 후속검색과 종료 로그

0건 검색에서 같은 세션의 다음 검색으로 이어지는 전이를 만듭니다. 종료 이벤트 수는 실제 이탈자 수와 다릅니다.

In [7]:
zero_with_next = base[base.zero & base.has_next].copy()
zero_with_next["transition_type"] = zero_with_next.apply(classify_transition, axis=1)
zero_with_next["recovered"] = zero_with_next.next_total_result_count.gt(0)
zero_with_next["detail_entered"] = zero_with_next.next_search_id.isin(click_search_ids)

card_b = {
    "followup_after_zero": metric(len(zero_with_next), int(base.zero.sum()), "zero-result search"),
    "search_sessions": int(base.session_id.nunique()),
    "session_start_events": int(event.event_type.eq("session_start").sum()),
    "session_end_events": int(event.event_type.eq("session_end").sum()),
    "immediate_exit_rate_calculable": False,
}
show_metrics({'0건 후 후속검색': card_b['followup_after_zero']})
display(pd.DataFrame([{'검색 세션': card_b['search_sessions'], '시작 이벤트': card_b['session_start_events'], '종료 이벤트': card_b['session_end_events']}]))
display(zero_with_next[['search_id','next_search_id','transition_type','recovered','detail_entered']].head(10))

,지표,분자,분모,비율(%),단위
0,0건 후 후속검색,3271,3434,95.25,zero-result search


,검색 세션,시작 이벤트,종료 이벤트
0,1000,1000,23


,search_id,next_search_id,transition_type,recovered,detail_entered
0,SYN_Q0001_001,SYN_Q0001_002,조건 강화,False,False
1,SYN_Q0001_002,SYN_Q0001_003,동일조건 반복,False,False
2,SYN_Q0001_003,SYN_Q0001_004,검색어 변경,False,False
3,SYN_Q0001_004,SYN_Q0001_005,지역 변경,True,False
7,SYN_Q0002_003,SYN_Q0002_004,조건 강화,False,False
14,SYN_Q0005_001,SYN_Q0005_002,조건 강화,False,False
15,SYN_Q0005_002,SYN_Q0005_003,동일조건 반복,False,False
16,SYN_Q0005_003,SYN_Q0005_004,지역 변경,False,False
17,SYN_Q0005_004,SYN_Q0005_005,동일조건 반복,False,False
18,SYN_Q0005_005,SYN_Q0005_006,검색어 변경,False,False


## 6. 카드 C — 즉시 회복과 세션 회복

즉시 회복: 다음 검색이 양수인 전이/0건 후 전이. 기존 JSON의 세션 회복: 0건 뒤 어느 시점이든 양수 검색이 존재한 세션/0건 경험 세션입니다.

**정의 확인:** 첨부 카드의 “마지막 검색이 양수”와 기존 코드의 “0건 뒤 한 번이라도 양수”는 일반적으로 다른 정의입니다. 아래에서 두 값을 함께 확인합니다. 서로 다른 분모의 즉시·세션 회복률을 상승률로 비교하지 않습니다.

In [8]:
final_numerator = 0
final_denominator = 0
for _, group in base.groupby("session_id", sort=False):
    values = group.total_result_count.astype(int).tolist()
    zero_positions = [index for index, value in enumerate(values) if value == 0]
    if zero_positions:
        final_denominator += 1
        final_numerator += any(
            any(value > 0 for value in values[index + 1 :])
            for index in zero_positions
        )
card_c = {
    "immediate_recovery": metric(int(zero_with_next.recovered.sum()), len(zero_with_next), "zero-to-next transition"),
    "session_final_recovery": metric(final_numerator, final_denominator, "session experiencing any zero result"),
}
show_metrics(card_c)
zero_sessions = base.groupby('session_id')['zero'].any()
last_results = base.groupby('session_id').tail(1).set_index('session_id')['total_result_count']
last_positive_n = int(last_results.loc[zero_sessions[zero_sessions].index].gt(0).sum())
show_metrics({'마지막 검색 양수(별도 정의)': metric(last_positive_n, final_denominator, 'session')})
print('두 정의의 분자 차이:', final_numerator - last_positive_n)

,지표,분자,분모,비율(%),단위
0,immediate_recovery,558,3271,17.06,zero-to-next transition
1,session_final_recovery,488,651,74.96,session experiencing any zero result


,지표,분자,분모,비율(%),단위
0,마지막 검색 양수(별도 정의),488,651,74.96,session


두 정의의 분자 차이: 0


## 7. 카드 D·F — 재검색 방법별 회복과 상세진입

회복 분자: 다음 검색 결과가 양수인 전이. 상세진입 분자: 다음 검색에 `hotel_click`이 있는 전이. 분모는 각 방법의 전이 수입니다. 반복 클릭은 검색 단위에서 한 번으로 처리합니다.

In [9]:
method_results = {}
method_order = ["동일조건 반복", "조건 완화", "검색어 변경", "지역 변경", "조건 강화", "완화·강화 혼합"]
for name in method_order:
    group = zero_with_next[zero_with_next.transition_type.eq(name)]
    method_results[name] = {
        "transitions": len(group),
        "recovery": metric(int(group.recovered.sum()), len(group), "zero-to-next transition"),
        "detail_entry": metric(int(group.detail_entered.sum()), len(group), "zero-to-next transition"),
    }
method_table = []
for name, item in method_results.items():
    method_table.append({'방법': name, '전이 수': item['transitions'],
                         '회복 분자': item['recovery']['numerator'], '회복률(%)': item['recovery']['rate']*100,
                         '상세진입 분자': item['detail_entry']['numerator'], '상세진입률(%)': item['detail_entry']['rate']*100})
display(pd.DataFrame(method_table).style.format({'회복률(%)':'{:.2f}', '상세진입률(%)':'{:.2f}'}))

,방법,전이 수,회복 분자,회복률(%),상세진입 분자,상세진입률(%)
0,동일조건 반복,1238,0,0.00,0,0.00
1,조건 완화,959,256,26.69,187,19.50
2,검색어 변경,234,69,29.49,69,29.49
3,지역 변경,563,233,41.39,70,12.43
4,조건 강화,231,0,0.00,0,0.00
5,완화·강화 혼합,46,0,0.00,0,0.00


수치 해석: 지역 변경 회복 233/563과 검색어 변경 상세진입 69/234를 구분해서 읽습니다. 복제된 전이가 늘어났어도 원래 검색어 변경 표본 10건의 정보량이 늘어난 것은 아닙니다.

## 8. 카드 G — 클릭 기준 세션 분류

기존 보고서의 축약 정의를 재현합니다. 첫 검색에 클릭이 귀속되면 직접 성공, 이후 검색에 최초 클릭이 귀속되면 재검색 후 성공, 클릭이 없으면 상호작용 없음입니다.

이 분류는 찜·객실 선택을 포함하지 않으며 클릭 시각이 다음 검색보다 앞서는지, 조건을 실제 바꿨는지는 추가 검증하지 않습니다. 따라서 첨부 카드 G의 전체 정의와 동등하다고 보지 않습니다.

In [10]:
search_order = base[["session_id", "search_id", "search_order"]]
clicked_order = search_order[search_order.search_id.isin(click_search_ids)]
first_clicked_order = clicked_order.groupby("session_id").search_order.min().to_dict()
segments = {"직접 상호작용 성공": 0, "재검색 후 상호작용 성공": 0, "상호작용 없음": 0}
for session_id in base.session_id.unique():
    first_order = first_clicked_order.get(session_id)
    if first_order == 1:
        segments["직접 상호작용 성공"] += 1
    elif first_order is not None:
        segments["재검색 후 상호작용 성공"] += 1
    else:
        segments["상호작용 없음"] += 1
card_g = {
    name: metric(value, int(base.session_id.nunique()), "session")
    for name, value in segments.items()
}
show_metrics(card_g)

,지표,분자,분모,비율(%),단위
0,직접 상호작용 성공,69,1000,6.90,session
1,재검색 후 상호작용 성공,630,1000,63.00,session
2,상호작용 없음,301,1000,30.10,session


## 9. 카드 H — 순위별 고유 클릭

분자: 한 번 이상 클릭된 고유 `(search_id, hotel_id)`. 분모: 해당 순위의 결과 행. 반복 클릭은 제거합니다. 여기서는 기존 보고서의 기술통계만 재현하며 가격·평점·편의시설 통제 순위모형은 실행하지 않습니다.

In [11]:
card_h = {}
for rank in range(1, 6):
    ranked = search_result[search_result.result_rank.eq(rank)]
    numerator = sum(
        pair in click_pairs
        for pair in ranked[["search_id", "hotel_id"]].itertuples(index=False, name=None)
    )
    card_h[str(rank)] = metric(numerator, len(ranked), "exposed search-result pair")
show_metrics(card_h)

,지표,분자,분모,비율(%),단위
0,1,680,3466,19.62,exposed search-result pair
1,2,255,3183,8.01,exposed search-result pair
2,3,139,2717,5.12,exposed search-result pair
3,4,69,2576,2.68,exposed search-result pair
4,5,93,2528,3.68,exposed search-result pair


## 10. A-2 — 대표 경로와 통제모형

동일 조건 시퀀스의 세션은 하나의 대표 세션으로 줄입니다. 날짜는 생성 중 교정되어 지문에서 제외합니다. 이는 원본 계보 ID를 검증한 복원이 아닌 시퀀스 기반 추정입니다. 43개 경로·296검색이 재현되는지 확인합니다.

모형: 0건 여부 ~ 가격 설정 + 평점 설정 + 편의시설 3개 이상 + 도시 + 체크인월 + 인원 + 숙박일수. 세션 군집 표준오차를 사용해 보고서의 추정값을 재현합니다. 동일 원본 사용자가 여러 세션을 가졌을 가능성 및 원본 생성 규칙 영향은 남아 있으므로 p값은 확정적 근거로 사용하지 않습니다.

In [12]:
signatures = {}
for session_id, group in base.groupby("session_id", sort=True):
    signatures[session_id] = session_signature(group)
signature_counts = pd.Series(signatures).value_counts()
seen = set()
representative_sessions = []
for session_id in sorted(signatures):
    signature = signatures[session_id]
    if signature not in seen:
        seen.add(signature)
        representative_sessions.append(session_id)
model_data = base[base.session_id.isin(representative_sessions)].copy()
model_data["zero_result"] = model_data.zero.astype(int)
model_data["price_set"] = model_data.price.notna().astype(int)
model_data["rating_set"] = model_data.user_rating_min.notna().astype(int)
model_data["amenity_ge3"] = model_data.amenity_count.ge(3).astype(int)
destination = model_data.destination.fillna("UNKNOWN").astype(str)
model_data["city"] = destination.str.extract(
    r"^(Tokyo|Osaka|Kyoto|Sapporo|Fukuoka)", expand=False
).fillna("UNKNOWN")
model_data["checkin_month"] = pd.to_datetime(model_data.checkin_date).dt.strftime("%Y-%m")
model_data["stay_nights"] = (
    pd.to_datetime(model_data.checkout_date) - pd.to_datetime(model_data.checkin_date)
).dt.days.clip(lower=1)
display(signature_counts.value_counts().sort_index().rename_axis('경로별 복제 수').to_frame('경로 수'))
print('대표 세션:', model_data.session_id.nunique(), '대표 검색:', len(model_data))
display(model_data[['zero_result','price_set','rating_set','amenity_ge3','city','checkin_month','guest_count','stay_nights']].head())

,경로 수
경로별 복제 수,
23,32
24,11


대표 세션: 43 대표 검색: 296


,zero_result,price_set,rating_set,amenity_ge3,city,checkin_month,guest_count,stay_nights
0,1,0,0,0,Osaka,2026-10,2,2
1,1,1,1,1,Osaka,2026-10,2,2
2,1,1,1,1,Osaka,2026-10,2,2
3,1,1,1,1,Osaka,2026-10,2,2
4,0,1,0,0,Osaka,2026-10,2,2


In [13]:
formula = (
    "zero_result ~ price_set + rating_set + amenity_ge3 + "
    "C(city) + C(checkin_month) + guest_count + stay_nights"
)
fitted = smf.glm(formula, data=model_data, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": model_data.session_id}
)
constraint_terms = {}
confidence = fitted.conf_int()
for term in ["price_set", "rating_set", "amenity_ge3"]:
    condition_off = model_data.copy()
    condition_on = model_data.copy()
    condition_off[term] = 0
    condition_on[term] = 1
    probability_off = float(fitted.predict(condition_off).mean())
    probability_on = float(fitted.predict(condition_on).mean())
    constraint_terms[term] = {
        "odds_ratio": float(np.exp(fitted.params[term])),
        "odds_ratio_ci95_low": float(np.exp(confidence.loc[term, 0])),
        "odds_ratio_ci95_high": float(np.exp(confidence.loc[term, 1])),
        "p_value": float(fitted.pvalues[term]),
        "standardized_probability_if_off": probability_off,
        "standardized_probability_if_on": probability_on,
        "standardized_difference_pp": (probability_on - probability_off) * 100,
    }
print('모형식:', formula)
print('수렴:', fitted.converged, '검색 수:', fitted.nobs)
display(pd.DataFrame(constraint_terms).T)
display(fitted.summary())

모형식: zero_result ~ price_set + rating_set + amenity_ge3 + C(city) + C(checkin_month) + guest_count + stay_nights
수렴: True 검색 수: 296


,odds_ratio,odds_ratio_ci95_low,odds_ratio_ci95_high,p_value,standardized_probability_if_off,standardized_probability_if_on,standardized_difference_pp
price_set,2.256846,0.378424,13.459372,3.716438e-01,0.450248,0.551881,10.163274
rating_set,1.378619,0.332258,5.720216,6.582974e-01,0.477784,0.515460,3.767616
amenity_ge3,29.694840,8.877239,99.330833,3.709198e-08,0.212400,0.823250,61.085001


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            zero_result   No. Observations:                  296
Model:                            GLM   Df Residuals:                      282
Model Family:                Binomial   Df Model:                           13
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -108.18
Date:                Sun, 06 Sep 2026   Deviance:                       216.37
Time:                        02:03:14   Pearson chi2:                     348.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.4807
Covariance Type:              cluster                                         
===============================================================================================
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -0.7582      1.864     -0.407      0.684      -4.411       2.895
C(city)[T.Kyoto]                0.1576      0.800      0.197      0.844      -1.410       1.725
C(city)[T.Osaka]                0.2867      0.963      0.298      0.766      -1.600       2.174
C(city)[T.Sapporo]             -1.0033      1.137     -0.882      0.378      -3.232       1.225
C(city)[T.Tokyo]               -0.4161      0.900     -0.462      0.644      -2.180       1.348
C(city)[T.UNKNOWN]              0.4753      0.843      0.564      0.573      -1.177       2.127
C(checkin_month)[T.2026-10]    -0.6336      0.873     -0.726      0.468      -2.345       1.078
C(checkin_month)[T.2026-11]     0.6654      0.747      0.891      0.373      -0.798       2.129
C(checkin_month)[T.2026-12]     2.1765      0.817      2.663      0.008       0.575       3.778
price_set                       0.8140      0.911      0.893      0.372      -0.972       2.600
rating_set                      0.3211      0.726      0.442      0.658      -1.102       1.744
amenity_ge3                     3.3910      0.616      5.504      0.000       2.183       4.598
guest_count                    -0.5348      0.585     -0.915      0.360      -1.681       0.611
stay_nights                    -0.0201      0.354     -0.057      0.955      -0.713       0.673
===============================================================================================
"""

수치 해석: 표준화 확률은 각 대표 검색의 조건 플래그만 0 또는 1로 바꾼 모형 예측값을 평균한 것입니다. 오즈비 29.69는 확률이 29.69배라는 뜻이 아니며, +61.1%p도 실제 개입 효과가 아닙니다. 가격·평점의 신뢰구간은 넓습니다.

## 11. 보고서 JSON 자동 대조와 품질 검증

아래에서 모든 카드의 분자·분모·비율 및 A-2 계수·확률을 기존 JSON과 비교합니다. 정수는 정확히, 실수는 작은 오차 허용 범위로 대조합니다. 이 PASS는 기존 계산 재현을 뜻하며 지표 정의나 인과적 타당성을 승인하는 의미는 아닙니다.

In [14]:
checks = {
    "sqlite_integrity_ok": integrity == "ok",
    "search_rows_6900": counts["search"] == 6900,
    "session_rows_1000": base.session_id.nunique() == 1000,
    "search_filter_one_to_one": len(base) == counts["search"] == counts["search_filter"],
    "search_result_reconciliation": int(base.total_result_count.sum()) == counts["search_result"],
    "all_zero_transitions_classified": sum(item["transitions"] for item in method_results.values()) == len(zero_with_next),
    "session_segments_sum_1000": sum(item["numerator"] for item in card_g.values()) == 1000,
    "source_trajectory_signatures_43": len(seen) == 43,
    "representative_searches_296": len(model_data) == 296,
    "representative_zero_searches_147": int(model_data.zero.sum()) == 147,
    "model_converged": bool(fitted.converged),
}
source_hash_after = sha256(db)
checks["source_hash_unchanged"] = before_hash == source_hash_after
actual_sections = {'card_A': card_a, 'card_B': card_b, 'card_C': card_c,
                   'card_D_F': method_results, 'card_G': card_g, 'card_H': card_h,
                   'A2_terms': constraint_terms}
comparison = []
def compare_tree(actual, expected, path):
    if isinstance(expected, dict):
        for key, value in expected.items():
            compare_tree(actual[key], value, path + '.' + key)
    else:
        if isinstance(expected, float):
            passed = bool(np.isclose(actual, expected, rtol=1e-6, atol=1e-9))
        else:
            passed = actual == expected
        comparison.append({'항목': path, '재계산': actual, '보고서 JSON': expected, '일치': passed})
for name, actual in actual_sections.items():
    expected = reference['A2_adjusted_model']['constraint_terms'] if name == 'A2_terms' else reference[name]
    compare_tree(actual, expected, name)
comparison_df = pd.DataFrame(comparison)
display(comparison_df)
display(pd.DataFrame(list(checks.items()), columns=['QA 항목', '통과']))
connection.close()
assert all(checks.values()), '품질 검증 실패'
assert comparison_df['일치'].all(), '보고서 수치와 다른 항목이 있습니다.'
print(f"보고서 대조 {len(comparison_df)}개 PASS / QA {len(checks)}개 PASS")
print('분석 전후 DB SHA-256:', source_hash_after)

,항목,재계산,보고서 JSON,일치
0,card_A.amenity_count_ge_3.numerator,2808,2808,True
1,card_A.amenity_count_ge_3.denominator,3181,3181,True
2,card_A.amenity_count_ge_3.rate,0.882741,0.882741,True
3,card_A.amenity_count_ge_3.analysis_unit,search,search,True
4,card_A.minimum_rating_set.numerator,2695,2695,True
...,...,...,...,...
130,A2_terms.amenity_ge3.odds_ratio_ci95_high,99.330833,99.330833,True
131,A2_terms.amenity_ge3.p_value,0.0,0.0,True
132,A2_terms.amenity_ge3.standardized_probability_...,0.2124,0.2124,True
133,A2_terms.amenity_ge3.standardized_probability_...,0.82325,0.82325,True


,QA 항목,통과
0,sqlite_integrity_ok,True
1,search_rows_6900,True
2,session_rows_1000,True
3,search_filter_one_to_one,True
4,search_result_reconciliation,True
5,all_zero_transitions_classified,True
6,session_segments_sum_1000,True
7,source_trajectory_signatures_43,True
8,representative_searches_296,True
9,representative_zero_searches_147,True


보고서 대조 135개 PASS / QA 12개 PASS
분석 전후 DB SHA-256: d30f40b9d0c85a8f8469f48b37daae5f6dc3e4a77eea81671310dda7cbd8e79d


## 12. 직접 확인하는 예시

`base`는 검색별 데이터, `zero_with_next`는 0건 후 전이, `model_data`는 대표 검색, `comparison_df`는 대조표입니다. 아래 필터를 바꾸어 개별 행을 확인할 수 있습니다.

In [15]:
display(zero_with_next.loc[zero_with_next['transition_type'].eq('지역 변경'),
    ['session_id','search_id','next_search_id','destination','next_destination','recovered','detail_entered']].head(20))

,session_id,search_id,next_search_id,destination,next_destination,recovered,detail_entered
3,SYN_S0001,SYN_Q0001_004,SYN_Q0001_005,Osaka · 우메다,Osaka,True,False
16,SYN_S0005,SYN_Q0005_003,SYN_Q0005_004,NaN,Tokyo · 신주쿠,False,False
20,SYN_S0005,SYN_Q0005_007,SYN_Q0005_008,Tokyo · 신주쿠,Tokyo · 시부야,False,False
51,SYN_S0013,SYN_Q0013_005,SYN_Q0013_006,Tokyo · 긴자·도쿄역,Osaka · 혼마치,True,False
59,SYN_S0014,SYN_Q0014_006,SYN_Q0014_007,Osaka · 베이 에어리어,NaN,True,True
77,SYN_S0017,SYN_Q0017_011,SYN_Q0017_012,Tokyo · 아사쿠사,Tokyo,True,False
93,SYN_S0019,SYN_Q0019_011,SYN_Q0019_012,Osaka · 우메다,Osaka · 난바,False,False
94,SYN_S0019,SYN_Q0019_012,SYN_Q0019_013,Osaka · 난바,Osaka · 혼마치,False,False
95,SYN_S0019,SYN_Q0019_013,SYN_Q0019_014,Osaka · 혼마치,Osaka · 베이 에어리어,False,False
97,SYN_S0019,SYN_Q0019_015,SYN_Q0019_016,Osaka · 베이 에어리어,Osaka · 신사이바시,False,False
